# TLE fitting with propygator

This notebook walks through **Feature 1.2**: re-expressing a reference orbit — a
high-fidelity numerical propagation, a user-assembled trajectory, or another TLE's
output — as a **shareable TLE**, via Orekit's batch least squares over the SGP4 mean
elements (+ B\*). `fit_tle` returns the fitted `TLE`; its sibling `fit_tle_detailed`
(identical parameters, one shared engine) returns a `FitResult` with the fit
diagnostics.

This is the faithful sibling of `TLE.from_state_unfitted` (notebook 04): that one
stuffs osculating elements into mean-element slots and is deliberately *not*
round-trip-faithful; `fit_tle` actually fits. The defining caveat, worth stating up
front: **the fit is inherently lossy.** SGP4 is a simplified model (J2/J3/J4 zonals
plus a single B\* drag term for the near-Earth branch), so a full-force numerical
orbit can never be reproduced exactly — expect a few hundred meters RMS over a 2-day
LEO span. We will *see* that number below, not just assert it.

We use a **pinned ISS element set** so the notebook runs offline and reproducibly.
As everywhere in propygator, building the `TLE` and the value types is pure-Python;
the JVM starts lazily at the first propagation (`docs/architecture.md` §10).

In [ ]:
import numpy as np

import propygator as pgr

pgr.__version__

## 1. A high-fidelity reference trajectory

The most common use of `fit_tle` is **path (a)** from `docs/architecture.md` §8:
you ran a careful numerical propagation and want to hand the result to tools that
only speak TLE. We build that reference here explicitly — 2 days of `leo_default`
physics (70x70 gravity, Sun/Moon third body, drag, SRP) from an ISS-like state,
with the `high_precision` integrator preset that §1.1 designed for exactly this.

(Passing a `State` to `fit_tle` runs this same internal propagation for you —
`fit_tle(state0, fitting_span=2 * 86400)` — but building the trajectory ourselves
lets us plot the divergence against it afterwards.)

In [ ]:
# A pinned ISS (ZARYA) element set (epoch 2026-06-20) anchors the scenario.
ISS_LINE1 = "1 25544U 98067A   26171.41461525  .00008813  00000+0  16600-3 0  9990"
ISS_LINE2 = "2 25544  51.6327 284.1189 0004557 208.5194 151.5545 15.49333088572250"

tle = pgr.TLE.from_strings(ISS_LINE1, ISS_LINE2, name="ISS (ZARYA)")

# The reference initial state: the TLE's own state at epoch, in EME2000.
state0 = pgr.propagate_tle(tle, 60, output_step=60)[0].to_frame(pgr.Frame.EME2000)

reference = pgr.propagate_numerical(
    state0,
    duration=2 * 86400,
    output_step=600,
    force_models=pgr.ForceModelConfig.leo_default(),
    spacecraft=pgr.SpacecraftConfig(),
    integrator=pgr.IntegratorConfig.high_precision(),
    progress=False,
)
print(len(reference), "samples over", reference.frame)

## 2. Fit the TLE

`fit_tle(reference, *, fitting_span=2 * 86400, ..., fit_bstar=True, norad_id=None,
name=None, progress=True)` seeds a template TLE at the reference start (a
fixed-point osculating-to-mean inversion of the first sample), then runs a
Levenberg-Marquardt batch least squares over ~300 evenly subsampled position/velocity
measurements. The fitted epoch is the **reference start**, and with
`fit_bstar=True` (the default — right for LEO, where a 2-day span makes drag
observable) B\* is estimated alongside the six mean elements.

Identity fields (catalog number, name, designator...) are bookkeeping no trajectory
can supply: they resolve as *explicit kwarg -> inherited from `initial_guess` ->
placeholder*. Physics is fitted-or-zeroed — the mean-motion derivatives are always
`0.0` (SGP4 ignores them).

By default the fit streams `iter N | rms ...` progress lines to stderr; we pass
`progress=False` to keep the notebook tidy.

In [ ]:
fitted = pgr.fit_tle(reference, norad_id=25544, name="ISS (ZARYA)", progress=False)

print("fitted:", fitted.line1, fitted.line2, sep="\n        ")
print("source:", tle.line1, tle.line2, sep="\n        ")

The fitted line 2 sits close to the source element set (same orbit, after all) but
not on it: the fitted elements describe **this 2-day full-force arc**, not the
catalog's radar fit. Note in particular B\* on line 1 — the fitted value is a *fit
residual* that absorbs the drag of this arc under SGP4's crude drag model, not a
physical ballistic coefficient (and not the catalog's `16600-3`).

## 3. The lossiness, made visible

The fitted TLE is a *shareable approximation*. Propagate it back over the fitted
span and plot the position error against the numerical reference — this is the
honest cost of the SGP4 re-expression.

In [ ]:
back = pgr.propagate_tle(
    fitted, 2 * 86400, output_step=600, start=reference.start_epoch
).to_frame(pgr.Frame.EME2000)

divergence_m = np.linalg.norm(back.positions - reference.positions, axis=1)
rms_m = float(np.sqrt(np.mean(divergence_m**2)))
print(f"propagate-back residual: {rms_m:.0f} m RMS, {divergence_m.max():.0f} m max")

In [ ]:
import matplotlib.pyplot as plt

hours = np.array(
    [s.epoch.seconds_since(reference.start_epoch) / 3600.0 for s in reference]
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hours, divergence_m, linewidth=1.5)
ax.set_xlabel("hours since fit epoch")
ax.set_ylabel("position error (m)")
ax.set_title("Fitted TLE vs the numerical reference (the lossiness)")
ax.grid(alpha=0.3)
fig.tight_layout()

A few hundred meters RMS over two days (the validation suite pins ~495 m RMS /
~1.1 km max for this scenario) — excellent by TLE standards, where against *reality*
errors are typically kilometers, but never zero: the oscillation is the full-force
physics SGP4 cannot represent. A longer `fitting_span` averages over more of the
unmodeled dynamics (lower peak error growth beyond the span, higher in-span
residual); a shorter one fits tighter but degrades faster outside it.

## 4. The fit diagnostics: `fit_tle_detailed`

`fit_tle` is a thin wrapper over `fit_tle_detailed`, which returns a frozen
`FitResult` carrying the fitted TLE plus the diagnostics: iterations, evaluations,
the final measurement RMS, and the **per-measurement position residuals** with
their epochs. The fit is deterministic, so re-running it reproduces the same TLE
exactly. (There is deliberately no `converged` flag — a non-converged fit raises
`TLEFitError` with no partial result, so a `FitResult` only exists for converged
fits.)

Note the distinction: `result.rms_m` is the observed-vs-estimated RMS over the
~300 *fit measurements* at convergence — the same number the final progress line
quotes — while the propagate-back curve above is the error over the full output
grid. For this noise-free reference they land close together.

In [ ]:
result = pgr.fit_tle_detailed(
    reference, norad_id=25544, name="ISS (ZARYA)", progress=False
)

assert result.tle == fitted  # one engine: fit_tle(...) is fit_tle_detailed(...).tle
print(
    f"converged in {result.iterations} iterations "
    f"({result.evaluations} evaluations), rms {result.rms_m:.0f} m "
    f"over {len(result.residuals_m)} measurements"
)

In [ ]:
meas_hours = np.array(
    [
        e.seconds_since(result.measurement_epochs[0]) / 3600.0
        for e in result.measurement_epochs
    ]
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(meas_hours, result.residuals_m, linewidth=1.5)
ax.set_xlabel("hours since fit epoch")
ax.set_ylabel("measurement residual (m)")
ax.set_title("Per-measurement residuals at convergence (FitResult)")
ax.grid(alpha=0.3)
fig.tight_layout()

## 5. The known-exact case: refitting SGP4's own output

When the reference *is* an SGP4 trajectory (**path (c)**), the fit has an exact
answer — and recovers it. This separates the estimation plumbing from the model
lossiness: the least squares is exact; SGP4's expressiveness is the only limit.
Passing the source TLE as `initial_guess` also donates its identity fields
(designator, element-set number, ...) to the fitted line.

In [ ]:
ref_sgp4 = pgr.propagate_tle(tle, 2 * 86400, output_step=60)
refit = pgr.fit_tle(ref_sgp4, initial_guess=tle, progress=False)

print("refit:  ", refit.line2)
print("source: ", tle.line2)

back_sgp4 = pgr.propagate_tle(
    refit, 2 * 86400, output_step=60, start=ref_sgp4.start_epoch
)
self_rms = float(
    np.sqrt(
        np.mean(np.linalg.norm(back_sgp4.positions - ref_sgp4.positions, axis=1) ** 2)
    )
)
print(f"self-fit propagate-back residual: {self_rms:.2g} m RMS")

## 6. Failure is honest

A fit that cannot converge within `max_iterations` raises `TLEFitError` — carrying
the iteration count and the last RMS, never a raw Java trace, and **no partial
TLE**. (Bad *inputs* — a non-positive `fitting_span`, too few samples, an unbound
orbit — raise `ValueError` pre-flight instead, before the JVM even starts.)

In [ ]:
try:
    pgr.fit_tle(ref_sgp4, max_iterations=1, progress=False)
except pgr.TLEFitError as exc:
    print("TLEFitError:", exc)

## Where to go next

- **The binding contract** — `docs/features.md` §1.2: the field policy (identity
  inherits, physics fitted-or-zeroed), the failure-mode table, the progress
  contract, and the as-built Outcome note with the validated numbers.
- **`fit_bstar=False`** holds B\* at the seed's value — pass it where drag is
  unobservable (short spans, GEO), so the estimate cannot wander off absorbing
  along-track error.
- **Deep space works too:** the SGP4/SDP4 branch follows automatically from the
  fitted mean motion, exactly as in `propagate_tle` (a Molniya self-fit recovers
  its source in the validation suite). LEO is the *validated* domain; elsewhere,
  `TLEFitError` is an honest outcome.
- **Progress reporting:** `progress=True` (default) streams per-iteration
  `iter N | rms ...` lines to stderr; a callable receives the fraction of the
  iteration budget consumed — try it from a terminal script.